<a href="https://colab.research.google.com/github/kriss-spy/pr-course-prj/blob/main/EvTrack/code/SDSTrack/SDSTrack_VisEvent_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDSTrack on VisEvent — Colab Notebook

This notebook reproduces **SDSTrack** evaluation on the **VisEvent** dataset for the Pattern Recognition course project (Topic #65).

## Workflow
1. **Environment Setup** — Mount Drive, install deps, clone SDSTrack, apply patches
2. **Dataset Preparation** — Extract VisEvent train/test sets from Google Drive
3. **Model Preparation** — Set up pretrained checkpoint, apply PyTorch 2.x compatibility fixes
4. **Evaluation** — Run tracking on test set and compute Success/Precision metrics

## Expected Results
- **Success AUC:** ~0.62
- **Precision (20px):** ~0.74

## Tips
- All setup cells are **idempotent** — safe to re-run.
- Long-running cells (extraction) show progress and have keepalive.
- Use *Runtime > Factory reset runtime* if you need a completely fresh start.

## 本 Notebook 解决什么问题？

**课题：** 基于事件相机的目标跟踪（Topic #65）

### 核心难点
传统帧相机在高速运动、极端光照变化等场景下存在运动模糊、信息延迟等问题。事件相机以像素级异步触发机制提供微秒级时间分辨率与高动态范围，但如何将其稀疏事件流与传统图像有效融合，是提升跟踪鲁棒性的关键。

### SDSTrack 复现痛点
- **环境断层**：原代码基于 PyTorch 1.11 + Python 3.8，Colab 已升级到 PyTorch 2.x + Python 3.10+
- **路径硬编码**：数据集、模型路径写死为作者本地服务器路径
- **checkpoint 兼容性**：PyTorch 2.6+ 默认 `weights_only=True`，无法加载旧模型
- **数据规模**：VisEvent 数据集 232 GB，需从 Google Drive 分卷解压并管理空间

### 本 Notebook 的解决方案
| 问题 | 解决方案 |
|------|----------|
| 环境不兼容 | 自动检测并应用 PyTorch 2.x / Python 3.10+ 兼容性补丁（`collections.abc`、`torch._six`、`weights_only` 等） |
| 路径错误 | 自动重写 `local.py` 和测试脚本中的硬编码路径为 Colab 路径 |
| 数据集管理 | 通过 Google Drive 挂载 + 7z 直接解压分卷，支持断点续传；自动清理 zip 释放空间 |
| 模型获取 | OSTrack 预训练模型和 SDSTrack checkpoint 通过 Drive 快捷方式绕过 Google 下载配额 |
| 长时间运行 | 评测单元格内置 keepalive 线程，防止 Colab 空闲断开；支持断点续跑 |
| 指标计算 | 自动读取 tracking results，计算 **Success AUC** 和 **Precision @ 20px**，直接用于报告 |

### 最终目标
在 VisEvent 测试集上复现 SDSTrack，获得量化的跟踪性能指标，为课程设计报告提供实验数据支撑。

In [ ]:
# ============================================================
# Quick State Check
# Run this anytime to see current progress.
# ============================================================

import os

def check(path):
    return os.path.exists(path)

def dir_items(path):
    if not check(path):
        return 0
    try:
        return len([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
    except:
        return 0

MOUNT = "/content/drive"
BASE = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
WS = "/content/sdstrack"

# Count test sequences (look inside wrapper dir if present)
test_seqs = 0
if check(f"{BASE}/test"):
    for d in os.listdir(f"{BASE}/test"):
        p = os.path.join(f"{BASE}/test", d)
        if os.path.isdir(p):
            test_seqs += dir_items(p)

checks = {
    "Google Drive mounted": check(f"{MOUNT}/MyDrive"),
    "SDSTrack cloned": check(f"{WS}/lib"),
    "Patches applied": check(f"{WS}/.patches_applied"),
    "Test zips present": check(f"{BASE}/VisEvent_dataset/VisEvent_test.zip") or check(f"{BASE}/VisEvent_test.zip"),
    "Train extracted": dir_items(f"{BASE}/train") > 0,
    "Test extracted": test_seqs > 0,
    "Pretrained model": check(f"{WS}/pretrained/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar"),
    "Eval model ready": check(f"{WS}/models/SDSTrack_cvpr2024_rgbe.pth.tar"),
}

print("=" * 50)
print("SDSTrack Setup State")
print("=" * 50)
for name, ok in checks.items():
    print(f"  {'✅' if ok else '⬜'} {name}")

print(f"  📁 Test sequences extracted: {test_seqs}")

done = sum(checks.values())
print("=" * 50)
print(f"Progress: {done}/{len(checks)}")
if done == len(checks):
    print("All set! Proceed to Phase 4: Evaluation.")
elif not checks["Google Drive mounted"]:
    print("Next: Run Phase 1 → Mount Google Drive")
elif not checks["SDSTrack cloned"]:
    print("Next: Run Phase 1 → Environment Setup")
elif not checks["Test extracted"]:
    print("Next: Run Phase 2 → Extract Test Set")
elif not checks["Eval model ready"]:
    print("Next: Run Phase 3 → Model Preparation")
else:
    print("Next: Continue with remaining setup steps.")

SDSTrack Setup State
  ⬜ Google Drive mounted
  ⬜ SDSTrack cloned
  ⬜ Patches applied
  ⬜ Test zips present
  ⬜ Train extracted
  ⬜ Test extracted
  ⬜ Pretrained model
  ⬜ Eval model ready
  📁 Test sequences extracted: 0
Progress: 0/8
Next: Run Phase 1 → Mount Google Drive


## Phase 1: Environment Setup

Set up the SDSTrack codebase and its dependencies.

**Notes:**
- Colab provides PyTorch 2.x (upstream requires 1.11.0, but we apply compatibility patches).
- All cells are idempotent — safe to re-run.

In [ ]:
# ============================================================
# 1.1 Mount Google Drive
# Idempotent: skips if already mounted.
# ============================================================

import os
from google.colab import drive

MOUNT_POINT = "/content/drive"

if os.path.ismount(MOUNT_POINT) and os.path.exists(f"{MOUNT_POINT}/MyDrive"):
    print("✅ Google Drive already mounted.")
else:
    print("🔌 Mounting Google Drive...")
    drive.mount(MOUNT_POINT, force_remount=False)
    print("✅ Drive mounted successfully!")

🔌 Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!


In [ ]:
# Post-lunch diagnostic
import os

BASE = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
test_dir = os.path.join(BASE, "test")

print(f"VisEvent base exists: {os.path.exists(BASE)}")

if os.path.exists(test_dir):
    seqs = 0
    for d in os.listdir(test_dir):
        p = os.path.join(test_dir, d)
        if os.path.isdir(p):
            for sub in os.listdir(p):
                if os.path.isdir(os.path.join(p, sub)):
                    seqs += 1
    print(f"Test sequences in Drive: {seqs}")
else:
    print("test/ directory not found in Drive")

# Check zip files
zip_dir = os.path.join(BASE, "VisEvent_dataset")
if os.path.exists(zip_dir):
    zips = sorted([f for f in os.listdir(zip_dir) if f.startswith("VisEvent_test")])
    print(f"Zip files available: {len(zips)}")
    for z in zips:
        sz = os.path.getsize(os.path.join(zip_dir, z)) / 1e9
        print(f"  {z} ({sz:.1f} GB)")


VisEvent base exists: True
Test sequences in Drive: 70
Zip files available: 6
  VisEvent_test.z01 (15.7 GB)
  VisEvent_test.z02 (18.9 GB)
  VisEvent_test.z03 (18.9 GB)
  VisEvent_test.z04 (18.9 GB)
  VisEvent_test.z05 (18.9 GB)
  VisEvent_test.zip (7.4 GB)


In [ ]:
# ============================================================
# 1.2 GPU and CUDA Verification
# Idempotent: pure diagnostics.
# ============================================================

import torch

print("=" * 50)
print("GPU / CUDA Info")
print("=" * 50)

!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

print("\nPyTorch CUDA Info")
print("-" * 50)
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA (PyTorch):   {torch.version.cuda}")
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Quick GPU tensor test
if torch.cuda.is_available():
    x = torch.rand(500, 500).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU tensor test:  OK (shape {y.shape})")
else:
    print("\n⚠️ WARNING: CUDA not available!")

GPU / CUDA Info
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-bddba83d-6bb9-8d11-89ac-1ee0dd563b0a)
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07

PyTorch CUDA Info
--------------------------------------------------
PyTorch version:  2.11.0+cu128
CUDA available:   True
CUDA (PyTorch):   12.8
GPU:              NVIDIA A100-SXM4-40GB
GPU memory:       42.41 GB

GPU tensor test:  OK (shape torch.Size([500, 500]))


In [ ]:
# ============================================================
# 1.3 Install SDSTrack Dependencies
# Idempotent: checks each package before installing.
# ============================================================

import subprocess
import sys
import importlib

print("Checking dependencies...")

PKGS = [
    ('PyYAML', 'yaml'),
    ('easydict', 'easydict'),
    ('cython', 'cython'),
    ('opencv-python', 'cv2'),
    ('pandas', 'pandas'),
    ('pycocotools', 'pycocotools'),
    ('jpeg4py', 'jpeg4py'),
    ('scipy', 'scipy'),
    ('timm==0.5.4', 'timm'),
    ('tb-nightly', 'tensorboard'),
    ('lmdb', 'lmdb'),
    ('visdom', 'visdom'),
    ('wandb', 'wandb'),
    ('vot-toolkit==0.5.3', 'vot_toolkit'),
    ('vot-trax==3.0.3', 'vot_trax'),
    ('tqdm', 'tqdm'),
]

missing = []
for pkg, mod in PKGS:
    try:
        importlib.import_module(mod.replace('-', '_'))
        print(f"  ✅ {pkg.split('==')[0]}")
    except ImportError:
        print(f"  ⬜ {pkg.split('==')[0]}")
        missing.append(pkg)

if missing:
    print(f"\nInstalling {len(missing)} missing packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ Installation complete.")
else:
    print("\n✅ All dependencies already installed.")

Checking dependencies...
  ✅ PyYAML
  ✅ easydict
  ✅ cython
  ✅ opencv-python
  ✅ pandas
  ✅ pycocotools
  ⬜ jpeg4py
  ✅ scipy
  ✅ timm
  ✅ tb-nightly
  ⬜ lmdb
  ⬜ visdom
  ✅ wandb
  ⬜ vot-toolkit
  ⬜ vot-trax
  ✅ tqdm

Installing 5 missing packages...
✅ Installation complete.


In [ ]:
# ============================================================
# 1.4 Verify Installation
# Idempotent: pure diagnostics.
# ============================================================

import torch
import cv2
import yaml
import timm
import scipy

print("=" * 50)
print("Dependency Versions")
print("=" * 50)
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()} ({torch.version.cuda if torch.cuda.is_available() else 'N/A'})")
print(f"OpenCV:   {cv2.__version__}")
print(f"timm:     {timm.__version__}")
print(f"scipy:    {scipy.__version__}")

if torch.cuda.is_available():
    x = torch.rand(1000, 1000).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU test: OK (result {y.shape})")
else:
    print("\n⚠️ No GPU detected!")

Dependency Versions
PyTorch:  2.11.0+cu128
CUDA:     True (12.8)
OpenCV:   4.13.0
timm:     1.0.27
scipy:    1.16.3

GPU test: OK (result torch.Size([1000, 1000]))


In [ ]:
# ============================================================
# 1.5 Clone SDSTrack & Apply Compatibility Patches
# Idempotent: skips clone/patch if already done.
# ============================================================

import os

WS = "/content/sdstrack"
os.makedirs(WS, exist_ok=True)
os.chdir(WS)

UPSTREAM = os.path.join(WS, ".upstream_cloned")
PATCHED = os.path.join(WS, ".patches_applied")
PATHS = os.path.join(WS, ".paths_fixed")

# --- Clone ---
if os.path.exists(UPSTREAM) and os.path.exists(os.path.join(WS, "lib")):
    print("✅ SDSTrack already cloned.")
else:
    print("🔽 Cloning SDSTrack...")
    !git clone https://github.com/hoqolo/SDSTrack.git .
    with open(UPSTREAM, "w") as f:
        f.write("done")
    print("✅ Clone complete.")

# --- Apply PyTorch 2.x / Python 3.10+ patches ---
if os.path.exists(PATCHED):
    print("✅ Patches already applied.")
else:
    loader = os.path.join(WS, "lib", "train", "data", "loader.py")
    if not os.path.exists(loader):
        print("❌ loader.py not found. Clone may have failed.")
    else:
        print("🔧 Applying compatibility patches...")
        with open(loader, "r") as f:
            c = f.read()

        if "import collections.abc" not in c:
            c = c.replace("import collections", "import collections\nimport collections.abc")

        c = c.replace("from torch._six import string_classes",
                      "try:\n    from torch._six import string_classes\nexcept ImportError:\n    string_classes = (str, bytes)")
        c = c.replace("collections.Mapping", "collections.abc.Mapping")
        c = c.replace("collections.Sequence", "collections.abc.Sequence")

        with open(loader, "w") as f:
            f.write(c)

        with open(PATCHED, "w") as f:
            f.write("done")
        print("✅ Patches applied to loader.py")

# --- Fix paths ---
if os.path.exists(PATHS):
    print("✅ Paths already fixed.")
else:
    print("⚙️ Configuring paths...")
    !python tracking/create_default_local_file.py --workspace_dir . --data_dir ./data --save_dir ./output

    local_py = os.path.join(WS, "lib", "train", "admin", "local.py")
    if os.path.exists(local_py):
        with open(local_py, "r") as f:
            c = f.read()
        c = c.replace(
            "self.visevent_dir = '/home/houxiaojun/Workspace/SDSTrack/data/visevent/train'",
            "self.visevent_dir = '/content/sdstrack/data/visevent/train/train_subset'"
        )
        with open(local_py, "w") as f:
            f.write(c)
        print("✅ Fixed visevent_dir")

    test_script = os.path.join(WS, "RGBE_workspace", "test_rgbe_mgpus.py")
    if os.path.exists(test_script):
        with open(test_script, "r") as f:
            c = f.read()
        c = c.replace(
            "seq_home = '/public/datasets_neo/VisEvent/VisEvent_dataset/testset/test_subset'",
            "seq_home = '/content/sdstrack/data/visevent/test/test_subset'"
        )
        with open(test_script, "w") as f:
            f.write(c)
        print("✅ Fixed test script path")

    with open(PATHS, "w") as f:
        f.write("done")
    print("✅ Paths configured.")

🔽 Cloning SDSTrack...
Cloning into '.'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 188 (delta 27), reused 184 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 3.30 MiB | 28.20 MiB/s, done.
Resolving deltas: 100% (27/27), done.
✅ Clone complete.
🔧 Applying compatibility patches...
✅ Patches applied to loader.py
⚙️ Configuring paths...
2026-06-02 03:51:21.341661: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 03:51:21.411136: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F

In [ ]:
# ============================================================
# 1.6 Test Imports
# Idempotent: verifies the environment is functional.
# ============================================================

import sys
sys.path.insert(0, "/content/sdstrack")

tests = [
    ("lib.train.data.loader", "LTRLoader"),
    ("lib.test.evaluation", "create_default_local_file_test"),
    ("lib.train.admin.local", "EnvironmentSettings"),
]

all_ok = True
for mod, obj in tests:
    try:
        m = __import__(mod, fromlist=[obj])
        getattr(m, obj)
        print(f"  ✅ {mod}.{obj}")
    except Exception as e:
        print(f"  ❌ {mod}.{obj}: {e}")
        all_ok = False

if all_ok:
    from lib.train.admin import local
    print(f"\n📁 VisEvent train dir: {local.EnvironmentSettings().visevent_dir}")
    print("\n🎉 Environment ready!")
else:
    print("\n⚠️ Some imports failed. Check earlier cells.")

  ✅ lib.train.data.loader.LTRLoader
  ✅ lib.test.evaluation.create_default_local_file_test
  ✅ lib.train.admin.local.EnvironmentSettings

📁 VisEvent train dir: /content/sdstrack/data/visevent/train

🎉 Environment ready!


In [ ]:
# ============================================================
# 1.7 Create Dataset Symlinks
# Idempotent: handles existing directories/links safely.
# ============================================================

import os
import shutil

WS = "/content/sdstrack"
DATA_DIR = os.path.join(WS, "data")
os.makedirs(DATA_DIR, exist_ok=True)

DRIVE_DATA = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
link = os.path.join(DATA_DIR, "visevent")

if os.path.islink(link) and os.path.exists(link):
    print(f"✅ Symlink already exists -> {os.readlink(link)}")
elif os.path.isdir(link) and not os.path.islink(link):
    print("⚠️ Directory exists, replacing with symlink...")
    shutil.rmtree(link)
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
        print(f"✅ Symlink created -> {DRIVE_DATA}")
    else:
        print(f"❌ Drive data not found: {DRIVE_DATA}")
elif not os.path.exists(link):
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
        print(f"✅ Symlink created -> {DRIVE_DATA}")
    else:
        print(f"❌ Drive data not found: {DRIVE_DATA}")

# Summary
print("\n📂 Data directory:")
for item in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, item)
    kind = "[LINK]" if os.path.islink(p) else "[DIR]" if os.path.isdir(p) else "[FILE]"
    print(f"  {item:20s} {kind}")

⚠️ Directory exists, replacing with symlink...
✅ Symlink created -> /content/drive/MyDrive/EvTrack/datasets/VisEvent

📂 Data directory:
  .gitkeep             [FILE]
  depthtrack           [DIR]
  lasher               [DIR]
  visevent             [LINK]


In [ ]:
# ============================================================
# 1.8 Download OSTrack Pretrained Model
# Idempotent: skips if model already exists.
#
# NOTE: The public gdown link often hits quota. Instead:
#  1. Open https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy?usp=sharing
#  2. Right-click 'OSTrack_ep0300.pth.tar' → Add shortcut → My Drive
#  3. Re-run this cell (reads shortcut from mounted Drive)
# ============================================================

import os
import shutil

PRETRAINED_DIR = "/content/sdstrack/pretrained/vitb_256_mae_ce_32x4_ep300"
expected = os.path.join(PRETRAINED_DIR, "OSTrack_ep0300.pth.tar")
shortcut = "/content/drive/MyDrive/OSTrack_ep0300.pth.tar"

if os.path.exists(expected):
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"✅ Model already exists ({mb:.1f} MB)")
elif os.path.exists(shortcut):
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    shutil.copy2(shortcut, expected)
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"✅ Copied from Drive shortcut ({mb:.1f} MB)")
else:
    print("⬜ Model not found.")
    print("\nPlease add a shortcut to your Google Drive:")
    print("  1. Open: https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy?usp=sharing")
    print("  2. Right-click 'OSTrack_ep0300.pth.tar'")
    print("  3. Select 'Organize' → 'Add shortcut' → 'All locations' → 'My Drive' → 'Add'")
    print("  4. Re-run this cell")

✅ Copied from Drive shortcut (354.2 MB)


## Phase 2: Dataset Preparation

Extract the VisEvent dataset from Google Drive zip archives.

**Prerequisite:** Dataset zip files must be in `MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset/`.

| Set | Files | Size | Est. Time | Needed? |
|-----|-------|------|-----------|---------|
| Train | `VisEvent_train.zip` + `.z01`–`.z06` | ~130 GB | 2–4 h | **No** (training cancelled) |
| Test  | `VisEvent_test.zip` + `.z01`–`.z05` | ~102 GB | 1–3 h | **Yes** (evaluation) |

We use `7z` to extract directly from split parts (no merge step needed).

**Important:**
- Only the **test set** is required since training reproduction was cancelled (Issue #4).
- If your previous extraction was interrupted, the old idempotent check may have falsely marked it as complete. Re-run the test extraction cell after reading the notes below.
- Keep the Colab tab active to prevent idle disconnect.
- Run the keepalive cell (2.3b) in parallel during long extraction.

In [ ]:
# ============================================================
# CLEANUP: Delete partial extraction and start fresh
# Run this ONCE before Cell 2.3 if previous extraction failed.
# ============================================================

import os
import shutil

TEST_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/test"

if os.path.exists(TEST_DIR):
    print(f"🗑️ Deleting partial extraction: {TEST_DIR}")
    shutil.rmtree(TEST_DIR)
    print("✅ Deleted.")
else:
    print("⬜ No partial extraction found.")

# Verify deletion
print(f"\nTest dir exists after cleanup: {os.path.exists(TEST_DIR)}")

🗑️ Deleting partial extraction: /content/drive/MyDrive/EvTrack/datasets/VisEvent/test
✅ Deleted.

Test dir exists after cleanup: False


In [ ]:
#!/usr/bin/env python3
"""
VisEvent Test Set Re-Downloader for Colab

Downloads VisEvent test set (6 zip parts, ~102 GB) from Dropbox
to Google Drive via Dropbox sharing API. Supports resume.

Usage in Colab:
    export DROPBOX_TOKEN="sl.xxx..."
    python visevent_test_downloader.py
"""

import os
import sys
import json
import time
import requests
from pathlib import Path
from google.colab import userdata
# =============================================
# CONFIG
# =============================================
DROPBOX_TOKEN = userdata.get('DROPBOX_TOKEN')

SHARED_LINK = "https://www.dropbox.com/scl/fo/r406wsgll56fy0hhhwu62/AFo3cjXjSI4Dzjn5nlnXNW0?rlkey=ecgyd26j1ycfl1jbm4pwc3vbn&st=rzf95buf&dl=0"

# Test set files to download
TEST_FILES = [
    "VisEvent_test.zip",
    "VisEvent_test.z01",
    "VisEvent_test.z02",
    "VisEvent_test.z03",
    "VisEvent_test.z04",
    "VisEvent_test.z05",
]

DEST_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
CHUNK_SIZE = 50 * 1024 * 1024  # 50 MB per chunk


def get_remote_size(filename: str) -> int | None:
    """Get file size from Dropbox without downloading body."""
    dropbox_path = f"/VisEvent_dataset/{filename}"
    headers = {
        "Authorization": f"Bearer {DROPBOX_TOKEN}",
        "Dropbox-API-Arg": json.dumps({"url": SHARED_LINK, "path": dropbox_path}),
    }
    resp = requests.post(
        "https://content.dropboxapi.com/2/sharing/get_shared_link_file",
        headers=headers,
        stream=True,
        timeout=30,
    )
    if resp.status_code in (200, 206):
        return int(resp.headers.get("Content-Length", 0))
    return None


def download_file_via_api(filename: str) -> bool:
    """Download a single file from Dropbox shared link via API.
    Robust retry: auto-resumes from break on ChunkedEncodingError/ConnectionError.
    """
    dest_path = os.path.join(DEST_DIR, filename)
    dropbox_path = f"/VisEvent_dataset/{filename}"
    os.makedirs(DEST_DIR, exist_ok=True)

    # 1. Get remote size
    remote_size = get_remote_size(filename)
    if remote_size is None:
        print(f"  ❌ Could not get remote size for {filename}")
        return False

    # 2. Check local state
    local_size = os.path.getsize(dest_path) if os.path.exists(dest_path) else 0

    if local_size == remote_size:
        print(f"  ✅ {filename}: already complete ({local_size / 1e9:.2f} GB)")
        return True
    elif local_size > 0:
        print(f"  ⚠️  Partial file ({local_size / 1e9:.2f}/{remote_size / 1e9:.2f} GB). Resuming from byte {local_size}...")
    else:
        print(f"  📥 Downloading {filename} ({remote_size / 1e9:.2f} GB)...")

    # 3. Robust download with retry loop
    max_retries = 10
    retry_delay = 5  # seconds, doubles each retry
    downloaded = local_size
    last_print = time.time()
    total_size = remote_size

    print(f"  Progress: ", end="", flush=True)

    for attempt in range(max_retries):
        if attempt > 0:
            print(f"\n  🔄 Retry {attempt}/{max_retries} after {retry_delay}s...")
            time.sleep(retry_delay)
            retry_delay = min(retry_delay * 2, 120)

        headers = {
            "Authorization": f"Bearer {DROPBOX_TOKEN}",
            "Dropbox-API-Arg": json.dumps({"url": SHARED_LINK, "path": dropbox_path}),
        }

        if downloaded > 0:
            headers["Range"] = f"bytes={downloaded}-"
            mode = "ab"
        else:
            mode = "wb"

        resp = None
        try:
            resp = requests.post(
                "https://content.dropboxapi.com/2/sharing/get_shared_link_file",
                headers=headers,
                stream=True,
                timeout=300,  # 5 min timeout per chunk read
            )

            if resp.status_code not in (200, 206):
                print(f"\n  ❌ HTTP {resp.status_code}")
                resp.close()
                continue  # Will retry

            with open(dest_path, mode) as f:
                for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                    if not chunk:
                        break
                    f.write(chunk)
                    downloaded += len(chunk)

                    now = time.time()
                    if now - last_print > 10:
                        pct = downloaded / total_size * 100
                        print(f"\r  Progress: {pct:.1f}% ({downloaded / 1e9:.2f}/{total_size / 1e9:.2f} GB)", end="", flush=True)
                        last_print = now

            # Success! Check if complete
            if downloaded >= total_size:
                print(f"\r  ✅ {filename}: complete ({downloaded / 1e9:.2f} GB)          ")
                return True
            else:
                print(f"\n  ⚠️  Download ended early ({downloaded}/{total_size} bytes). Will retry.")
                # Loop will continue to next attempt

        except requests.exceptions.ChunkedEncodingError as e:
            print(f"\n  ⚠️  Connection broken: {e}")
            print(f"  💾 Saved {downloaded / 1e9:.2f} GB. Will resume...")
            # Continue to retry
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            print(f"\n  ⚠️  Network error: {e}")
            print(f"  💾 Saved {downloaded / 1e9:.2f} GB. Will resume...")
            # Continue to retry
        except Exception as e:
            print(f"\n  ⚠️  Unexpected error: {e}")
            print(f"  💾 Saved {downloaded / 1e9:.2f} GB. Will resume...")
            # Continue to retry
        finally:
            if resp is not None:
                try:
                    resp.close()
                except:
                    pass

    print(f"\n  ❌ {filename}: FAILED after {max_retries} retries")
    print(f"     Downloaded {downloaded / 1e9:.2f}/{total_size / 1e9:.2f} GB")
    return False


def main():
    if DROPBOX_TOKEN in ("YOUR_TOKEN_HERE", ""):
        print("ERROR: Set DROPBOX_TOKEN environment variable.")
        print("  export DROPBOX_TOKEN='sl.xxx...'")
        sys.exit(1)

    print("=" * 60)
    print("VisEvent Test Set Re-Downloader")
    print("=" * 60)
    print(f"Destination: {DEST_DIR}")
    print(f"Files to download: {len(TEST_FILES)}")
    print("")

    os.makedirs(DEST_DIR, exist_ok=True)

    success = 0
    for filename in TEST_FILES:
        print(f"\n[{success+1}/{len(TEST_FILES)}] {filename}")
        if download_file_via_api(filename):
            success += 1
        else:
            print(f"  FAILED. You can re-run this script to retry.")

    print(f"\n{'='*60}")
    print(f"Downloaded: {success}/{len(TEST_FILES)} files")
    if success == len(TEST_FILES):
        print("✅ All test set files ready!")
        print(f"👉 Next: Run Cell 2.3 to extract the test set.")
    else:
        print("⚠️  Some files failed. Re-run this script to retry.")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


VisEvent Test Set Re-Downloader
Destination: /content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset
Files to download: 6


[1/6] VisEvent_test.zip
  ✅ VisEvent_test.zip: already complete (7.40 GB)

[2/6] VisEvent_test.z01
  ⚠️  Partial file (15.73/18.87 GB). Resuming from byte 15728640000...
  ✅ VisEvent_test.z01: complete (18.87 GB)          

[3/6] VisEvent_test.z02
  📥 Downloading VisEvent_test.z02 (18.87 GB)...
  Progress: 4.4% (0.84/18.87 GB)
  ⚠️  Connection broken: ('Connection broken: IncompleteRead(844185600 bytes read, 18030182400 more expected)', IncompleteRead(844185600 bytes read, 18030182400 more expected))
  💾 Saved 0.84 GB. Will resume...

  🔄 Retry 1/10 after 5s...
  ✅ VisEvent_test.z02: complete (18.87 GB)          

[4/6] VisEvent_test.z03
  📥 Downloading VisEvent_test.z03 (18.87 GB)...
  ✅ VisEvent_test.z03: complete (18.87 GB)          

[5/6] VisEvent_test.z04
  📥 Downloading VisEvent_test.z04 (18.87 GB)...
  ✅ VisEvent_test.z04: complete (18.87 GB)    

### 2.1 Extract VisEvent Train Set

Extracts to `MyDrive/EvTrack/datasets/VisEvent/train/`.

If the cell seems stuck with no output, check Drive for growing folders — 7z does not print per-file progress.

In [ ]:
# ============================================================
# 2.1 Extract VisEvent Train Set — SKIPPED
# Training reproduction was cancelled (Issue #4).
# Only the test set is needed for evaluation.
# ============================================================

print("ℹ️  Training reproduction cancelled (Issue #4).")
print("ℹ️  Only the TEST set is required for evaluation.")
print("ℹ️  Skipping train set extraction to save time and space.")
print("\n👉 Proceed to Cell 2.3 to extract the test set.")

ℹ️  Training reproduction cancelled (Issue #4).
ℹ️  Only the TEST set is required for evaluation.
ℹ️  Skipping train set extraction to save time and space.

👉 Proceed to Cell 2.3 to extract the test set.


In [ ]:
# ============================================================
# 2.2 Clean Up Train Zip Files
# Idempotent: skips if already deleted or directory missing.
# ============================================================

import os

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
ALT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"

# Find zip files in either location
found_files = []
for d in [VISEVENT_DIR, ALT_DIR]:
    if os.path.exists(d):
        found_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.startswith("VisEvent_train")])

if not found_files:
    print("✅ No train zip files to delete.")
else:
    total = sum(os.path.getsize(f) / 1e9 for f in found_files)
    print(f"🗑️ Deleting {len(found_files)} files ({total:.1f} GB)...")
    for f in found_files:
        os.remove(f)
        print(f"   ✓ {os.path.basename(f)}")
    print("\n✅ Cleanup complete.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset'

In [ ]:
# ============================================================
# 3.2b Fix Config Path (additional patch)
# The sdstrack parameter file hardcodes the experiment yaml path.
# ============================================================

import os

WS = "/content/sdstrack"
config_py = os.path.join(WS, "lib", "config", "sdstrack", "config.py")

if os.path.exists(config_py):
    with open(config_py, "r") as f:
        c = f.read()

    old = "/home/houxiaojun/Workspace/SDSTrack/experiments/"
    new = "/content/sdstrack/experiments/"

    if old in c:
        c = c.replace(old, new)
        with open(config_py, "w") as f:
            f.write(c)
        print(f"✅ Patched config path: {old} -> {new}")
    else:
        print("⬜ Config path already patched or not found.")
else:
    print(f"❌ Config file not found: {config_py}")

# Also fix the parameter file if it has a hardcoded path
param_py = os.path.join(WS, "lib", "test", "parameter", "sdstrack.py")
if os.path.exists(param_py):
    with open(param_py, "r") as f:
        c = f.read()

    if old in c:
        c = c.replace(old, new)
        with open(param_py, "w") as f:
            f.write(c)
        print(f"✅ Patched parameter path: {old} -> {new}")
    else:
        print("⬜ Parameter path already patched or not found.")

⬜ Config path already patched or not found.
⬜ Parameter path already patched or not found.


In [ ]:
# ============================================================
# 1.5b Fix local.py paths
# The local.py file was created with upstream hardcoded paths.
# ============================================================

import os

WS = "/content/sdstrack"
local_py = os.path.join(WS, "lib", "test", "evaluation", "local.py")

if os.path.exists(local_py):
    with open(local_py, "r") as f:
        content = f.read()

    old_path = "/home/houxiaojun/Workspace/SDSTrack"
    if old_path in content:
        content = content.replace(old_path, WS)
        with open(local_py, "w") as f:
            f.write(content)
        print(f"✅ Fixed local.py paths: {old_path} -> {WS}")
    else:
        print("⬜ local.py paths already correct or not found.")
else:
    print("⚠️ local.py does not exist. Re-running create_default_local_file.py...")
    !python /content/sdstrack/tracking/create_default_local_file.py --workspace_dir /content/sdstrack --data_dir ./data --save_dir ./output
    print("✅ Created local.py with correct paths.")

# Verify
from lib.test.evaluation.environment import env_settings
es = env_settings()
print(f"\nVerification:")
print(f"  prj_dir:  {es.prj_dir}")
print(f"  save_dir: {es.save_dir}")


✅ Fixed local.py paths: /home/houxiaojun/Workspace/SDSTrack -> /content/sdstrack

Verification:
  prj_dir:  /home/houxiaojun/Workspace/SDSTrack
  save_dir: /home/houxiaojun/Workspace/SDSTrack/output


In [ ]:
# Quick check: is test data accessible?
import os
TEST_DIR = "/content/sdstrack/data/visevent/test/test_subset"
print(f"Test dir exists: {os.path.exists(TEST_DIR)}")
# Just check one known sequence instead of listing all
known_seq = os.path.join(TEST_DIR, "00141_tank_outdoor2")
print(f"Known seq exists: {os.path.exists(known_seq)}")
if os.path.exists(known_seq):
    print("✅ Test data is accessible.")
    print("   (Skipping full count to avoid Drive latency)")
else:
    print("❌ Test data not found.")

Test dir exists: False
Known seq exists: False
❌ Test data not found.


### 2.3 Extract VisEvent Test Set

Extracts to `MyDrive/EvTrack/datasets/VisEvent/test/`.

Same procedure as train set.

In [ ]:
# ============================================================
# Evaluation Progress Check
# Run this periodically to see how many sequences are done.
# ============================================================

import os
import time

RESULTS_DIR = "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
TEST_DIR = "/content/sdstrack/data/visevent/test/test_subset"

if os.path.exists(TEST_DIR):
    total = len([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))])
else:
    total = 0

done = 0
if os.path.exists(RESULTS_DIR):
    done = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])

print(f"Evaluation progress: {done}/{total} sequences completed ({done/total*100:.1f}%)")
print(f"Results dir: {RESULTS_DIR}")

# Show last 5 completed sequences
if done > 0:
    files = sorted(os.listdir(RESULTS_DIR))
    print(f"\nLast 5 completed:")
    for f in files[-5:]:
        print(f"  {f}")


Evaluation progress: 8/302 sequences completed (2.6%)
Results dir: /content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe

Last 5 completed:
  00236_tennis_outdoor4.txt
  00241_tennis_outdoor4.txt
  00282_tennis_outdoor4.txt
  00292_tennis_outdoor4.txt
  00297_tennis_outdoor4.txt


In [ ]:
# ============================================================
# 2.3 Extract VisEvent Test Set
# CRITICAL FIX: Never auto-deletes existing data.
# The actual Dropbox distribution contains ~302 test sequences,
# not the 320 claimed in the paper. If ≥300 sequences exist,
# extraction is skipped. Otherwise, it runs and may overwrite.
# ============================================================

import os
import subprocess

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
TEST_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/test"

# Count existing sequences
actual_seqs = 0
if os.path.exists(TEST_DIR):
    for d in os.listdir(TEST_DIR):
        p = os.path.join(TEST_DIR, d)
        if os.path.isdir(p):
            for sub in os.listdir(p):
                if os.path.isdir(os.path.join(p, sub)):
                    actual_seqs += 1

if actual_seqs >= 300:
    print(f"✅ Test set already extracted ({actual_seqs} sequences).")
    print("   Skipping extraction. Proceed to Phase 3.")
    raise SystemExit

print(f"⬜ Test set not yet extracted ({actual_seqs} sequences found).")
print(f"   Will extract from zip archives.")
print(f"   NOTE: The complete distribution has ~302 sequences.")
print("")

# Discover zip files
zip_dirs = [VISEVENT_DIR, "/content/drive/MyDrive/EvTrack/datasets/VisEvent"]
found_dir = None
zips = []
for d in zip_dirs:
    if os.path.exists(d):
        z = [f for f in os.listdir(d) if f.startswith("VisEvent_test")]
        if z:
            found_dir = d
            zips = z
            break

if not zips:
    print("❌ No test zip files found.")
    print(f"   Checked: {zip_dirs}")
    raise SystemExit

print(f"📦 Found {len(zips)} test zip files in: {found_dir}")

# Ensure 7z is available
try:
    subprocess.run(["7z"], capture_output=True, check=True)
except Exception:
    print("📦 Installing p7zip-full...")
    !apt-get update -qq && apt-get install -y -qq p7zip-full

os.makedirs(TEST_DIR, exist_ok=True)

print("=" * 50)
print("Extracting VisEvent TEST set")
print("=" * 50)
print(f"Source:      {found_dir}")
print(f"Destination: {TEST_DIR}")
print("Expected:    ~302 sequences (complete distribution)")
print("Time:        1–3 hours.")
print("=" * 50)

result = subprocess.run(
    ["7z", "x", "-y", "VisEvent_test.zip", f"-o{TEST_DIR}"],
    cwd=found_dir,
    capture_output=False,
    text=True
)

# Verify
seqs = []
if os.path.exists(TEST_DIR):
    for d in os.listdir(TEST_DIR):
        p = os.path.join(TEST_DIR, d)
        if os.path.isdir(p):
            seqs.extend([s for s in os.listdir(p) if os.path.isdir(os.path.join(p, s))])

print("\n" + "=" * 50)
if result.returncode == 0:
    print(f"✅ Test extraction complete! ({len(seqs)} sequences)")
else:
    print(f"⚠️ 7z exited with code {result.returncode}")
    print(f"   Extracted {len(seqs)} sequences so far.")
    print(f"   This may indicate disk-space issue or corrupt download.")
print("=" * 50)

⬜ Test set not yet extracted (0 sequences found).
   Will extract from zip archives.
   NOTE: The complete distribution has ~302 sequences.

📦 Found 6 test zip files in: /content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset
Extracting VisEvent TEST set
Source:      /content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset
Destination: /content/drive/MyDrive/EvTrack/datasets/VisEvent/test
Expected:    ~302 sequences (complete distribution)
Time:        1–3 hours.

⚠️ 7z exited with code 2
   Extracted 256 sequences so far.
   This may indicate disk-space issue or corrupt download.


### 2.3b Keepalive

**Colab Limitation:** You cannot run this cell in parallel with Cell 2.3. Colab only runs one cell at a time.

**Alternative:** If you want to prevent Colab idle disconnect during the 1–3 hour extraction, use a browser console script or keep the Colab tab visible and active.

For a simple console keepalive (run in your browser's DevTools console on the Colab tab):

```javascript
function keepalive() {
  setInterval(() => {
    console.log('Keeping alive...');
    document.querySelector('colab-toolbar-button')?.dispatchEvent(new Event('click'));
  }, 60000);
}
keepalive();
```


In [ ]:
# ============================================================
# 2.3b Keepalive (runs standalone, NOT in parallel)
# Colab runs one cell at a time. Use this ONLY when no other
# cell is running, or use a browser console script instead.
# ============================================================

import time
from datetime import datetime

print("💓 Keepalive started. Will print every 60 seconds.")
print("   NOTE: This blocks the kernel. Do not run during extraction.")
print("   Use browser console keepalive instead (see cell above).")

for i in range(1, 1000):
    time.sleep(60)
    now = datetime.now().strftime("%H:%M:%S")
    print(f"[{now}] Keepalive #{i} — still alive...")

In [ ]:
# ============================================================
# 2.4 Clean Up Test Zip Files
# Idempotent: skips if already deleted or directory missing.
# ============================================================

import os

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
ALT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"

# Find zip files in either location
found_files = []
for d in [VISEVENT_DIR, ALT_DIR]:
    if os.path.exists(d):
        found_files.extend([os.path.join(d, f) for f in os.listdir(d) if f.startswith("VisEvent_test")])

if not found_files:
    print("✅ No test zip files to delete.")
else:
    total = sum(os.path.getsize(f) / 1e9 for f in found_files)
    print(f"🗑️ Deleting {len(found_files)} files ({total:.1f} GB)...")
    for f in found_files:
        os.remove(f)
        print(f"   ✓ {os.path.basename(f)}")
    print("\n✅ Cleanup complete.")

In [ ]:
# ============================================================
# 2.5 Verify Extracted Structure
# Idempotent: pure diagnostics.
# NOTE: The Dropbox distribution contains fewer sequences than
#       the paper claims. Actual counts: ~120 train, ~302 test.
# ============================================================

import os

BASE = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"

# The paper claims 500/320, but the actual Dropbox distribution
# is a curated subset. We use realistic thresholds.
EXPECTED_MIN = {
    "train": 100,   # actual ~120
    "test": 300,    # actual ~302
}

all_ok = True

for split in ["train", "test"]:
    split_dir = os.path.join(BASE, split)
    expected = EXPECTED_MIN[split]
    print(f"\n{'='*50}")
    print(f"{split.upper()} SET")
    print("="*50)

    if not os.path.exists(split_dir):
        print("⬜ Not yet extracted")
        continue

    top = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
    if not top:
        print("❌ No directories found")
        continue

    # Check for wrapper folders (train_subset / test_subset)
    first_top = os.path.join(split_dir, top[0])
    sub = [d for d in os.listdir(first_top) if os.path.isdir(os.path.join(first_top, d))]

    if sub:
        sequences = sub
        wrapper = top[0]
        print(f"📁 Wrapper:   {wrapper}")
    else:
        sequences = top
        wrapper = None
        print(f"📁 Direct structure")

    print(f"📁 Sequences: {len(sequences)} (expected ≥{expected})")

    if len(sequences) < expected:
        print(f"❌ INCOMPLETE: only {len(sequences)} sequences found.")
        all_ok = False
    else:
        print(f"✅ COMPLETE: {len(sequences)} sequences found.")

    if sequences:
        seq_path = os.path.join(split_dir, wrapper, sequences[0]) if wrapper else os.path.join(split_dir, sequences[0])
        print(f"\nExample '{sequences[0]}':")
        for exp in ["vis_imgs", "event_imgs", "groundtruth.txt"]:
            ok = os.path.exists(os.path.join(seq_path, exp))
            print(f"  {exp:20s} {'✅' if ok else '❌'}")

print("\n" + "="*50)
if all_ok:
    print("✅ Verification complete — data is ready for evaluation.")
else:
    print("⚠️  Verification complete — some sets are INCOMPLETE.")
print("="*50)

## Phase 3: Model Preparation

Prepare the pretrained checkpoint for evaluation and apply PyTorch 2.x compatibility patches.

In [ ]:
# ============================================================
# 3.1 Prepare Model Checkpoint
# Idempotent: skips if already set up.
# ============================================================

import os
import shutil

WS = "/content/sdstrack"
os.chdir(WS)

MODEL_DIR = os.path.join(WS, "models")
CKPT_DIR = os.path.join(WS, "output", "checkpoints", "train", "sdstrack", "cvpr2024_rgbe")
SYMLINK = os.path.join(MODEL_DIR, "SDSTrack_cvpr2024_rgbe.pth.tar")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists(SYMLINK):
    mb = os.path.getsize(os.path.realpath(SYMLINK)) / (1024 * 1024)
    print(f"✅ Model ready ({mb:.1f} MB)")
    print(f"   {SYMLINK}")
    raise SystemExit

# Search for shortcuts in Drive
NAMES = ["SDSTrack_cvpr2024_rgbe.pth.tar", "SDSTrack_ep0050.pth.tar"]
found = None
for name in NAMES:
    for path in [f"/content/drive/MyDrive/{name}", f"/content/drive/MyDrive/SDSTrack_models/{name}"]:
        if os.path.exists(path):
            found = path
            break
    if found:
        break

if found:
    dest = os.path.join(CKPT_DIR, "SDSTrack_ep0050.pth.tar")
    shutil.copy2(found, dest)
    os.symlink(dest, SYMLINK)
    mb = os.path.getsize(dest) / (1024 * 1024)
    print(f"✅ Copied from Drive ({mb:.1f} MB)")
    print(f"   {SYMLINK} -> {dest}")
else:
    print("⬜ Model checkpoint not found.")
    print("\nPlease add a shortcut to your Google Drive:")
    print("  1. Find the model file in the shared folder")
    print("  2. Right-click → 'Organize' → 'Add shortcut' → 'My Drive'")
    print("  3. Re-run this cell")

✅ Copied from Drive (489.0 MB)
   /content/sdstrack/models/SDSTrack_cvpr2024_rgbe.pth.tar -> /content/sdstrack/output/checkpoints/train/sdstrack/cvpr2024_rgbe/SDSTrack_ep0050.pth.tar


In [ ]:
# ============================================================
# 3.2 Apply PyTorch 2.x Compatibility Patches
# PyTorch 2.6+ defaults to weights_only=True, breaking old checkpoints.
# Idempotent: safe to re-run.
# ============================================================

import os

WS = "/content/sdstrack"
PATCHES = [
    ("lib/test/tracker/sdstrack.py",
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu")',
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu", weights_only=False)'),
    ("lib/train/trainers/base_trainer.py",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"),
]

for filepath, old, new in PATCHES:
    full = os.path.join(WS, filepath)
    if not os.path.exists(full):
        print(f"❌ File not found: {filepath}")
        continue

    with open(full, "r") as f:
        content = f.read()

    if old in content:
        content = content.replace(old, new)
        with open(full, "w") as f:
            f.write(content)
        print(f"✅ Patched {filepath}")
    else:
        print(f"⬜ Already patched or not found: {filepath}")

print("\nDone. Proceed to Phase 4: Evaluation.")

✅ Patched lib/test/tracker/sdstrack.py
✅ Patched lib/train/trainers/base_trainer.py

Done. Proceed to Phase 4: Evaluation.


## Phase 4: Evaluation

Run SDSTrack on the VisEvent test set and compute metrics.

**Expected time:** 30–90 minutes (single-threaded for stability).

**Output:**
- Tracking results: `RGBE_workspace/results/VisEvent/cvpr2024_rgbe/`
- Metrics: Success AUC, Precision @ 20px

In [ ]:
# ============================================================
# 4.1 Create testlist.txt and trainlist.txt
# Idempotent: skips if files already exist.
# Handles missing directories gracefully.
# ============================================================

import os

TEST_DIR = "/content/sdstrack/data/visevent/test/test_subset"
TRAIN_DIR = "/content/sdstrack/data/visevent/train/train_subset"

for dirname, listname in [(TEST_DIR, "testlist.txt"), (TRAIN_DIR, "trainlist.txt")]:
    path = os.path.join(dirname, listname)
    if os.path.exists(path):
        with open(path, "r") as f:
            n = len(f.readlines())
        print(f"✅ {listname} exists ({n} sequences)")
    elif not os.path.exists(dirname):
        print(f"⬜ {listname}: directory missing ({dirname})")
    else:
        seqs = sorted([d for d in os.listdir(dirname) if os.path.isdir(os.path.join(dirname, d))])
        with open(path, "w") as f:
            for s in seqs:
                f.write(s + "\n")
        print(f"✅ Created {listname} ({len(seqs)} sequences)")

✅ Created testlist.txt (302 sequences)
✅ trainlist.txt exists (120 sequences)


In [ ]:
# ============================================================
# 4.2 Run Evaluation on VisEvent Test Set
# - Streams output in real-time
# - threads=4 by default (safe on A100 40GB, V100 16GB)
#   Use threads=1 for T4 to avoid OOM.
# - Resumable: skips completed sequences (if upstream supports it)
# ============================================================

import os
import subprocess
import threading
import time

WS = "/content/sdstrack"
os.chdir(WS)

MODEL_PATH = "./models/SDSTrack_cvpr2024_rgbe.pth.tar"
if not os.path.exists(MODEL_PATH):
    print("❌ Model not found. Run Phase 3 first.")
    raise SystemExit

# Detect GPU and choose threads
gpu_name = ""
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
except:
    pass

if "A100" in gpu_name:
    threads = 4
elif "V100" in gpu_name:
    threads = 4
elif "L4" in gpu_name:
    threads = 4
else:
    threads = 1  # T4 safe default

print(f"Detected GPU: {gpu_name if gpu_name else 'unknown'}")
print(f"Using threads={threads}")

# Dynamically count total test sequences
TEST_DIR = "./data/visevent/test/test_subset"
total_seqs = 0
if os.path.exists(TEST_DIR):
    for d in os.listdir(TEST_DIR):
        p = os.path.join(TEST_DIR, d)
        if os.path.isdir(p):
            total_seqs += 1

if total_seqs == 0:
    print("❌ No test sequences found. Run Phase 2 first.")
    raise SystemExit

RESULTS_DIR = "./RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
existing = 0
if os.path.exists(RESULTS_DIR):
    existing = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])
    if existing > 0:
        print(f"⏮️ Found {existing} existing result files")

print("=" * 50)
print("Running SDSTrack Evaluation")
print("=" * 50)
print(f"Model:    {MODEL_PATH}")
print(f"Test set: {TEST_DIR}")
print(f"Total sequences: {total_seqs}")
print(f"Threads:  {threads}")
if threads == 1:
    print("Expected: ~60–90 min")
else:
    print(f"Expected: ~{max(2, int(total_seqs * 2.5 / threads / 60))}–{max(4, int(total_seqs * 4.0 / threads / 60))} hours")
print("=" * 50)

# Keepalive thread (works because it's inside the running cell)
keepalive_running = True
def keepalive():
    start = time.time()
    while keepalive_running:
        time.sleep(30)
        elapsed = time.time() - start
        current = 0
        if os.path.exists(RESULTS_DIR):
            current = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])
        print(f"\n⏱️  [{elapsed/60:.1f} min] Completed: {current}/{total_seqs} (keepalive)\n", flush=True)

t = threading.Thread(target=keepalive, daemon=True)
t.start()

# Run evaluation
process = subprocess.Popen(
    ["python", "./RGBE_workspace/test_rgbe_mgpus.py",
     "--script_name", "sdstrack",
     "--num_gpus", "1",
     "--threads", str(threads),
     "--epoch", "50",
     "--yaml_name", "cvpr2024_rgbe"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted by user")

process.wait()
keepalive_running = False

final = 0
if os.path.exists(RESULTS_DIR):
    final = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])

print("\n" + "=" * 50)
if process.returncode == 0 and final >= total_seqs:
    print(f"✅ Evaluation complete! ({final}/{total_seqs} sequences)")
elif final > 0:
    print(f"⚠️ Partial results: {final}/{total_seqs} sequences")
    print("Re-run this cell to continue (completed sequences may be skipped).")
else:
    print(f"❌ Evaluation failed (exit code: {process.returncode})")
print(f"Results: {RESULTS_DIR}")
print("=" * 50)

FileNotFoundError: [Errno 2] No such file or directory: '/content/sdstrack'

In [ ]:
# ============================================================
# GPU Recommendation & Runtime Check
# Run this to see your current GPU and estimate time.
# ============================================================

import subprocess
import torch

print("=" * 60)
print("Current GPU Info")
print("=" * 60)
!nvidia-smi --query-gpu=name,memory.total,memory.used,temperature.gpu,utilization.gpu --format=csv,noheader,nounits

gpu_name = torch.cuda.get_device_name(0)
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nDetected: {gpu_name} ({mem_total:.1f} GB)")

# Estimate throughput
if "T4" in gpu_name:
    est_per_seq = 3.2  # min/seq based on your current run
    threads_safe = 2
elif "V100" in gpu_name:
    est_per_seq = 1.5
    threads_safe = 4
elif "A100" in gpu_name:
    est_per_seq = 0.8
    threads_safe = 6
elif "L4" in gpu_name:
    est_per_seq = 1.0
    threads_safe = 4
else:
    est_per_seq = 3.0
    threads_safe = 1

total_seqs = 302
remaining = total_seqs - 6  # you've done 6
est_hours = (remaining * est_per_seq) / threads_safe / 60

print(f"\n{'='*60}")
print(f"Time Estimate for {remaining} remaining sequences:")
print(f"  GPU:           {gpu_name}")
print(f"  Safe threads:  {threads_safe}")
print(f"  Per sequence:  ~{est_per_seq:.1f} min")
print(f"  Estimated:     ~{est_hours:.1f} hours")
print(f"{'='*60}")

print("\n💡 Recommendations:")
if "T4" in gpu_name:
    print("  1. With Colab Pro, try Runtime → Change runtime type → GPU")
    print("     (Pro gives higher chance of V100/A100 on reconnect)")
    print("  2. If you stay on T4, increase threads to 2 for ~2x speedup")
    print("     (memory headroom: ~8GB used / 16GB total)")
elif "V100" in gpu_name or "A100" in gpu_name or "L4" in gpu_name:
    print("  ✅ Great GPU! Consider threads=4 for max throughput.")
    print(f"  With {threads_safe} threads: ~{est_hours/2:.1f} hours estimated.")


Current GPU Info
Tesla T4, 15360, 0, 49, 0

Detected: Tesla T4 (15.6 GB)

Time Estimate for 296 remaining sequences:
  GPU:           Tesla T4
  Safe threads:  2
  Per sequence:  ~3.2 min
  Estimated:     ~7.9 hours

💡 Recommendations:
  1. With Colab Pro, try Runtime → Change runtime type → GPU
     (Pro gives higher chance of V100/A100 on reconnect)
  2. If you stay on T4, increase threads to 2 for ~2x speedup
     (memory headroom: ~8GB used / 16GB total)


In [ ]:
# ============================================================
# 4.3 Compute Evaluation Metrics
# Computes Success (AUC) and Precision (20px) from tracking results.
# ============================================================

import os
import numpy as np
import glob

RESULTS_DIR = "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
GT_BASE = "/content/sdstrack/data/visevent/test/test_subset"

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

def compute_metrics(results_dir, gt_base):
    if not os.path.exists(results_dir):
        print("❌ Results directory not found. Run Phase 4.2 first.")
        return

    files = sorted(glob.glob(os.path.join(results_dir, "*.txt")))
    if not files:
        print("❌ No result files found.")
        return

    print(f"Processing {len(files)} result files...")

    all_ious, all_dists = [], []

    for res_file in files:
        seq = os.path.basename(res_file).replace(".txt", "")
        gt_file = os.path.join(gt_base, seq, "groundtruth.txt")

        if not os.path.exists(gt_file):
            print(f"⚠️  Skipping {seq} (no groundtruth)")
            continue

        try:
            pred = np.loadtxt(res_file, delimiter=",")
            gt = np.loadtxt(gt_file, delimiter=",")
        except Exception:
            print(f"⚠️  Skipping {seq} (load error)")
            continue

        if pred.ndim == 1:
            pred = pred.reshape(1, -1)
        if gt.ndim == 1:
            gt = gt.reshape(1, -1)

        n = min(len(pred), len(gt))
        pred, gt = pred[:n], gt[:n]

        for p, g in zip(pred, gt):
            all_ious.append(compute_iou(p, g))
            pcx, pcy = p[0] + p[2]/2, p[1] + p[3]/2
            gcx, gcy = g[0] + g[2]/2, g[1] + g[3]/2
            all_dists.append(np.sqrt((pcx - gcx)**2 + (pcy - gcy)**2))

    if not all_ious:
        print("No valid data to compute metrics.")
        return

    all_ious = np.array(all_ious)
    all_dists = np.array(all_dists)

    # Success AUC
    thresholds = np.arange(0, 1.05, 0.05)
    success_rates = [np.mean(all_ious >= t) for t in thresholds]
    auc = np.mean(success_rates)

    # Precision @ 20px
    prec_20 = np.mean(all_dists <= 20)

    print("\n" + "=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Success AUC:      {auc:.4f}")
    print(f"Precision (20px): {prec_20:.4f}")
    print(f"Total frames:     {len(all_ious)}")
    print("=" * 50)

    return {"auc": auc, "prec_20": prec_20}

metrics = compute_metrics(RESULTS_DIR, GT_BASE)

## Appendix: Troubleshooting & Notes

### Common Issues

**1. `torch.load` weights_only error**
- Already fixed in Phase 3.2. If you see this error, re-run Phase 3.2.

**2. Missing `testlist.txt`**
- Already handled in Phase 4.1. The cell auto-creates it if missing.

**3. Evaluation crashes with multiprocessing**
- Phase 4.2 uses `threads=1` for stability. It is slower but reliable.

**4. Colab disconnects during long extraction**
- Keep the browser tab active.
- The evaluation cell has a built-in keepalive thread.

**5. Model not found**
- The OSTrack pretrained model and SDSTrack checkpoint must be added as **shortcuts** to your Google Drive (see Phase 1.8 and Phase 3.1).

### File Locations

| Item | Path |
|------|------|
| Workspace | `/content/sdstrack` |
| Data symlink | `/content/sdstrack/data/visevent` |
| Pretrained model | `/content/sdstrack/pretrained/...` |
| Checkpoint | `/content/sdstrack/output/checkpoints/...` |
| Results | `/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe/` |